# 第三部分：TensorRT 部署详解（NVIDIA GPU 极致推理）

TensorRT 是 NVIDIA 官方的 **高性能推理引擎**。它通过图优化、低精度计算和内存管理将模型编译为针对特定 GPU 架构高度优化的执行引擎，特别适合 **CV、OCR、Detection、LLM 推理**等需要低延迟和高吞吐的场景。

下面从 **原理 → 导出 ONNX → 构建 Engine → Python/C++ 推理 → 优化技巧 → 排错** 全面讲解。

---

## 一、TensorRT 核心概念

### 1.1 TensorRT 是什么

TensorRT 不是普通的模型格式转换工具，而是一个 **编译器 + 运行时**：

- **编译器（Builder）**：读取 ONNX 等格式的模型，经过大量图优化和 kernel 调优，生成序列化引擎（`.engine` / `.trt`）
- **运行时（Runtime）**：加载引擎，执行极速推理

核心技术：
- **Layer Fusion（层融合）**：将多个算子合并为一个 kernel，减少显存读写
- **Kernel Auto-Tuning**：在目标 GPU 上实测多种 CUDA kernel 实现，选最快的
- **Precision Calibration（精度校准）**：将 FP32 模型量化到 FP16/INT8，用少量校准数据保证精度
- **Memory Optimization**：重用中间张量显存，减少分配和碎片

### 1.2 核心术语

| 术语 | 含义 |
|------|------|
| **Engine** | 针对特定 GPU 和输入 profile 编译好的优化执行计划（可序列化为文件） |
| **Builder** | 负责从 ONNX 等输入构建 Engine，执行所有优化 pass |
| **Runtime** | 反序列化 Engine，提供推理接口 |
| **Context** | 执行上下文，持有中间显存、绑定输入输出，是实际推理的载体 |
| **Optimization Profile** | 动态 shape 的 min/opt/max 配置，决定 Engine 的尺寸适应范围 |
| **Calibration** | INT8 量化的校准过程，收集激活值范围以确定量化参数 |

### 1.3 TensorRT 完整工作流

```text
PyTorch Model (.pth)
      │  torch.onnx.export
      ▼
ONNX Model (.onnx)
      │  TensorRT Parser (Builder)
      ▼
TensorRT Engine (.engine / .trt)
      │  Runtime + Context
      ▼
极速推理（Python / C++）
```

**关键点**：TensorRT 不能直接加载 `.pth` 文件，必须通过 ONNX（或 TorchScript + torch-tensorrt）转换。

---

## 二、ONNX 导出 —— TensorRT 的入口

TensorRT 的 ONNX Parser 对 ONNX 模型有一定要求，导出时就要注意兼容性。

### 2.1 推荐导出配置

```python
import torch

model = YourModel().eval().cuda()
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    opset_version=13,              # TensorRT 8.x 兼容 13，9.x 推荐 17
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},    # 动态 batch
        "output": {0: "batch_size"}
    },
    do_constant_folding=True       # 折叠常量，简化图
)
```

**注意事项**：
- `opset_version` 要根据 TensorRT 版本选择：TensorRT 8.x 兼容 opset 13，TensorRT 9.x+ 推荐 17
- 动态 batch 必须用 `dynamic_axes` 标记，否则构建 Engine 时只能接受固定尺寸
- 导出前务必 `model.eval()`，并确保 dummy_input 在正确的 device 上

### 2.2 用 onnx-simplifier 优化 ONNX

TensorRT 对简洁的 ONNX 图解析更友好。导出后强烈建议用 `onnxsim` 简化：

```bash
pip install onnxsim
python -m onnxsim model.onnx model_sim.onnx
```

这能解决很多“明明 opset 支持，但 TensorRT 就是解析失败”的问题。

### 2.3 查 ONNX 算子兼容性

不是所有 ONNX 算子都被 TensorRT 支持。若遇到 `Unsupported operator` 错误，先查 [官方支持列表](https://github.com/onnx/onnx-tensorrt/blob/main/docs/operators.md)。

常见不支持的情况：
- 某些较新的激活函数（需升级 TensorRT 版本）
- 非标准卷积参数（如 padding 不对称、dilation 过大）
- NLP 模型中特殊的 Mask 操作（需等价替换）

---

## 三、trtexec 工具 —— 快速构建 Engine

TensorRT 自带的 `trtexec` 是构建、评测、调试引擎的利器，无需写一行 Python。

### 3.1 安装 TensorRT

- **Linux（推荐）**：下载 [NVIDIA TensorRT](https://developer.nvidia.com/tensorrt) 的 tar 包或 deb 包安装
- **Python API**：`pip install tensorrt` （确保与系统 TensorRT 版本一致）
- 同时需安装 `pycuda` 用于 Python 推理（`pip install pycuda`）

### 3.2 构建 FP32 Engine

```bash
trtexec --onnx=model_sim.onnx \
        --saveEngine=model_fp32.engine \
        --workspace=2048                # 单位为 MiB
```

### 3.3 构建 FP16 Engine

```bash
trtexec --onnx=model_sim.onnx \
        --fp16 \
        --saveEngine=model_fp16.engine \
        --workspace=2048
```

FP16 通常加速 1.5~3 倍，精度损失极小（适合绝大多数 CV 模型）。

### 3.4 构建 INT8 Engine

```bash
trtexec --onnx=model_sim.onnx \
        --int8 \
        --saveEngine=model_int8.engine \
        --calib=calibration.cache
```

INT8 需要**校准数据**（几百到几千张真实样本），`trtexec` 可以从目录读取图像自动校准。加速可达 3~8 倍，但需验证精度是否达标。

### 3.5 动态 Shape Engine

若 ONNX 用了 `dynamic_axes`，必须指定 min/opt/max shapes：

```bash
trtexec --onnx=model_sim.onnx \
        --minShapes=input:1x3x224x224 \
        --optShapes=input:8x3x224x224 \
        --maxShapes=input:32x3x224x224 \
        --saveEngine=model_dynamic.engine \
        --fp16
```

**参数含义**：
- `minShapes`：最小的输入尺寸（Engine 必须能处理）
- `optShapes`：TensorRT 将针对此尺寸深度优化（期望最常见的 batch）
- `maxShapes`：最大的输入尺寸（Engine 必须能处理，超出会报错）

**性能提示**：推理时实际 batch 越接近 `optShapes`，效率越高。若在 `min` 或 `max` 附近运行，性能会下降。

### 3.6 用 trtexec 直接测速

```bash
trtexec --loadEngine=model_fp16.engine \
        --shapes=input:8x3x224x224 \
        --iterations=1000
```

输出会显示：
- Throughput (QPS)
- Latency (mean, median, 99%)
- Device memory usage

---

## 四、TensorRT Python 推理（完整生产级示例）

手动管理 GPU 显存和 CUDA 流有一定复杂度，以下是一个可以直接用于生产环境的封装。

### 4.1 完整推理类

```python
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import numpy as np

class TRTInference:
    def __init__(self, engine_path, fp16=False):
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        
        # 加载序列化引擎
        with open(engine_path, "rb") as f:
            engine_data = f.read()
        self.engine = self.runtime.deserialize_cuda_engine(engine_data)
        self.context = self.engine.create_execution_context()
        
        # 获取绑定信息
        self.input_name = self.engine.get_tensor_name(0)
        self.output_name = self.engine.get_tensor_name(1)
        self.dtype = np.float16 if fp16 else np.float32
        
        # 预分配输出内存（假设已知输出尺寸）
        self.output_shape = None
        self.d_output = None
        self.stream = cuda.Stream()

    def infer(self, input_data: np.ndarray) -> np.ndarray:
        """
        input_data: numpy array, shape = (batch, C, H, W), dtype = float32
        返回: numpy array
        """
        # 输入数据转换
        if self.dtype == np.float16:
            input_data = input_data.astype(np.float16)
        batch_size = input_data.shape[0]
        
        # 设置动态 batch（如果 Engine 支持）
        input_shape = input_data.shape
        self.context.set_input_shape(self.input_name, input_shape)
        
        # 分配输入内存
        d_input = cuda.mem_alloc(input_data.nbytes)
        cuda.memcpy_htod_async(d_input, input_data, self.stream)
        
        # 分配输出内存（仅当 shape 变化时重新分配）
        output_shape = self.context.get_tensor_shape(self.output_name)
        if self.output_shape != output_shape:
            self.output_shape = output_shape
            self.d_output = cuda.mem_alloc(
                np.prod(output_shape) * self.dtype().itemsize
            )
        
        # 绑定并执行
        bindings = [int(d_input), int(self.d_output)]
        self.context.execute_async_v2(
            bindings=bindings,
            stream_handle=self.stream.handle
        )
        
        # 拷贝输出回主机
        output_data = np.empty(self.output_shape, dtype=self.dtype)
        cuda.memcpy_dtoh_async(output_data, self.d_output, self.stream)
        self.stream.synchronize()
        
        return output_data

    def __del__(self):
        del self.context
        del self.engine
```

### 4.2 使用示例

```python
# 加载引擎
infer_engine = TRTInference("model_fp16.engine", fp16=True)

# 准备数据
input_data = np.random.randn(4, 3, 224, 224).astype(np.float32)

# 推理
output = infer_engine.infer(input_data)
print("Output shape:", output.shape)
```

**生产环境要点**：
- `cuda.mem_alloc` 只在第一次或 shape 变化时分配，避免频繁 GPU 内存分配
- 使用 `cuda.Stream` 异步拷贝和执行，可与数据预处理/后处理重叠
- `__del__` 中显式释放 context 和 engine，防止显存泄漏

---

## 五、TensorRT 优化核心原理

### 5.1 Layer Fusion（层融合）

将图中连续多个算子合并为单一 CUDA kernel：

```
# 原始图
Conv → BatchNorm → ReLU

# 融合后
CBR (Conv + BatchNorm + ReLU)  一个 Kernel 完成
```

其他常见融合：`Conv + Bias`、`Conv + ElementWise`、`MatrixMultiply + Activate`。

融合减少了两件事：
1. **Kernel Launch 次数**（从 3 次变 1 次）
2. **显存读写**（无需保存中间结果到显存再读取）

### 5.2 Kernel Auto-Tuning

同一个算子（如卷积）可能存在多种 CUDA kernel 实现：
- **Implicit GEMM**（通用，适合小 batch）
- **Winograd**（大卷积核、小输入时极快）
- **cuDNN 的各种算法**

TensorRT 构建引擎时，会在目标 GPU 上实际运行这些候选 kernel，选择耗时最短的。这就是为什么构建 Engine 需要较长时间，但推理极快。

### 5.3 FP16 / INT8 精度分析

| 精度 | 速度提升 | 精度损失 | 适用场景 |
|------|----------|----------|----------|
| FP32 | 基准 (1×) | 无 | 精度要求极高的任务 |
| FP16 | 1.5 ~ 3× | 极小（<0.1%） | 绝大多数视觉模型 |
| INT8 | 3 ~ 8× | 小（<0.5%） | 需校准，端侧/高吞吐场景 |

**FP16 为什么快**：
- Tensor Core 指令吞吐是 FP32 的 8 倍（A100 上）
- 数据量减半，显存带宽压力降低

**INT8 校准原理**：
- 收集几千张真实图片在网络中各层的激活值分布
- 确定量化参数（scale、zero point），使得量化后信息损失最小
- 校准算法：MinMax、Entropy（KL 散度）、Percentile 等

### 5.4 动态 Shape 与 Memory Pooling

TensorRT 在构建 Engine 时根据 min/opt/max shape 预分配一个内存池。每次推理时，仅从池中选取合适的缓冲区，而不反复 `cudaMalloc`。这是动态 shape 模型也能快速推理的关键。

---

## 六、TensorRT 常见错误与排错

| 错误现象 | 可能原因 | 解决方案 |
|----------|----------|----------|
| `Failed to parse ONNX model` | 图中存在不支持的算子或结构 | 提升 opset、用 onnx-simplifier、替换不支持的算子 |
| `[TRT] building engine failed` | 动态 shape 未配置 min/opt/max | 检查 Optimization Profile 是否正确设置 |
| `out of memory` | maxShapes 过大导致预分配过多显存 | 减小 maxShapes 或降低 max workspace |
| `mismatch between allocated bound and requested shape` | 推理时输入尺寸超出 min/max 范围 | 确保输入在范围内，或重新构建 Engine |
| FP16/INT8 精度丢失严重 | 模型某些层对精度敏感 | 使用混合精度（部分层保持 FP32），检查校准数据集质量 |
| GPU 推理速度反而不如 CPU | batch 太小，kernel launch 开销占比过大 | 增大 batch，合并多个请求成批次 |

---

## 七、TensorRT 优缺点总结

| 优点 | 缺点 |
|------|------|
| 🚀 **极致性能**：FP16 通常 2× 提升，INT8 可达 8× | 🔒 **仅限 NVIDIA GPU** |
| 💾 **显存优化**：内存复用、池化 | ⏳ **构建 Engine 耗时长**（需数分钟到数十分钟） |
| 📐 **动态 Shape 支持** | 🐛 **算子兼容性有限**，部分模型需改造 |
| 🧩 **序列化引擎**：一次构建，重复加载 | 🔧 **调试困难**：图优化后难追踪 |

---

## 八、工业级部署架构

### 8.1 标准链路

```text
PyTorch 训练
      ↓
ONNX 导出 (带 dynamic_axes)
      ↓
trtexec / Python API 构建 FP16 Engine
      ↓
C++ / Python 推理服务
      ↓
Triton Inference Server (模型管理、动态批处理、负载均衡)
      ↓
Kubernetes + GPU 集群
```

### 8.2 选择决策表

| 场景 | 方案 |
|------|------|
| CPU 后台服务 | LibTorch / ONNX Runtime |
| 跨平台 / 边缘设备 | ONNX Runtime |
| **NVIDIA GPU 极致性能** | **TensorRT (FP16 / INT8)** |
| 大模型 (LLM) 推理 | TensorRT-LLM |
| 高并发在线推理 | Triton Server + TensorRT Engine |

---

## 九、三种方案全景对比与推荐学习路径

经过 LibTorch → ONNX → TensorRT 三部分讲解，现在可以画出完整的工业部署技术地图：

```
                     PyTorch 训练模型
                          │
                          ▼
                  TorchScript (model.pt)
                   /              \
                  /                \
         LibTorch (C++)         ONNX (model.onnx)
         适合 C++ 原生服务         │
                                  ▼
                          ONNX Runtime (通用推理)
                                  │
                                  ▼
                          TensorRT Engine (.engine)
                          极致 GPU 推理
                                  │
                                  ▼
                      Triton Inference Server
                          生产级推理平台
```

**三种方案定位总结**：

| | LibTorch | ONNX Runtime | TensorRT |
|--|----------|-------------|----------|
| 性能 | 中等 | 良好 | 极致 |
| 灵活性 | 最高 | 高 | 中（算子受限） |
| 部署难度 | 中 | 低 | 高 |
| 适用场景 | C++ 原生集成 | 跨平台 / 快速部署 | GPU 高吞吐服务 |

---
